# 🧠 CAR-IMU — Week 2 Pipeline
**Class-conditional Autoregressive IMU Token Generator**

This notebook runs the full Week 2 pipeline:
1. ✅ Sanity-check the CAR-IMU model architecture
2. 🏋️ Train the decoder on Bio-PM token sequences
3. ⚡ Generate synthetic tokens for minority activity classes
4. 📊 Evaluate: real-only vs augmented HAR classifier

---
> **Works on:** Local machine OR Google Colab (run the Colab Setup cell first if on Colab)

## 🌐 Colab Setup (skip if running locally)

In [ ]:
import os

ON_COLAB = 'google.colab' in str(get_ipython())

if ON_COLAB:
    print("Running on Google Colab — setting up...")

    # Mount Google Drive (optional — for saving checkpoints persistently)
    from google.colab import drive
    drive.mount('/content/drive')

    # Clone your repo
    REPO_URL = "https://github.com/aviral23032002/bio-pm-guided-synthetic-imu-generation.git"
    !git clone {REPO_URL} /content/project
    %cd /content/project

    # Install dependencies
    !pip install torch h5py scikit-learn matplotlib umap-learn --quiet

    # Download WISDM dataset (if needed)
    # !wget -q https://www.cis.fordham.edu/wisdm/includes/datasets/latest/WISDM_ar_v1.1.tar.gz
    # !tar -xzf WISDM_ar_v1.1.tar.gz

    print("✅ Colab setup complete!")
else:
    print("Running locally.")
    # Set working directory to project root
    PROJECT_ROOT = os.path.dirname(os.path.abspath("week2_car_imu.ipynb"))
    os.chdir(PROJECT_ROOT)
    print(f"Working directory: {os.getcwd()}")

---
## Step 0 — Check GPU / Device

In [ ]:
import torch

if torch.cuda.is_available():
    device = 'cuda'
    print(f"✅ CUDA GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = 'mps'
    print("✅ Apple Silicon GPU (MPS) — M1/M2/M3/M4")
else:
    device = 'cpu'
    print("⚠️  No GPU found — using CPU (training will be slower)")

print(f"Device: {device}")

---
## Step 1 — Architecture Sanity Check
Verifies the CAR-IMU model forward pass and autoregressive sampling work correctly.

In [ ]:
# Runs the built-in sanity check in car_imu_decoder.py
# Expected output:
#   Model parameters: ~137,472
#   MSE loss (random init): ~1.0
#   All 6 classes sample shape (192, 64)

!python car_imu_decoder.py

---
## Step 2 — Train the CAR-IMU Decoder

Trains the 4-layer causal transformer on all 10,810 Bio-PM token sequences.

**Expected results:**
- Epoch 1:  Train MSE ~0.8
- Epoch 10: Train MSE ~0.3
- Epoch 50: Train MSE ~0.02, Val MSE ~0.016 ✅

**Time:** ~10 min on MPS (M4 Mac) / ~5 min on CUDA / ~20 min on CPU

In [ ]:
# Train the CAR-IMU decoder
# Saves best checkpoint to car_imu_checkpoints/best_model.pt
# Saves loss curve to car_imu_checkpoints/loss_curve.png

!python train_car_imu.py \
    --token_store results_week1/token_store.hdf5 \
    --out_dir     car_imu_checkpoints \
    --epochs      50 \
    --device      auto

In [ ]:
# Plot the training loss curve
import numpy as np
import matplotlib.pyplot as plt

train_losses = np.load('car_imu_checkpoints/train_losses.npy')
val_losses   = np.load('car_imu_checkpoints/val_losses.npy')

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train MSE', color='#4361ee', linewidth=2)
plt.plot(val_losses,   label='Val MSE',   color='#f72585', linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('CAR-IMU Training Loss Curve')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best val MSE:   {val_losses.min():.4f} (epoch {val_losses.argmin()+1})")
print(f"Final train MSE: {train_losses[-1]:.4f}")

---
## Step 3 — Generate Synthetic Tokens

Uses the trained CAR-IMU to generate synthetic token sequences for minority classes.

**Strategy options:**
- `upsample` — bring all classes to majority class count (~14K synthetic, ~20 min)
- `ratio 0.5` — add 50% extra for minority classes only (~1.6K synthetic, ~2 min) ← recommended first

**Temperature:** 0.5 = recommended. Lower = less noise, higher = more variety.

In [ ]:
# OPTION A: Quick test with ratio strategy (recommended to try first)
# Generates ~1,640 synthetic windows for minority classes only
# Time: ~2 minutes on MPS

!python generate_synthetic.py \
    --checkpoint  car_imu_checkpoints/best_model.pt \
    --token_store results_week1/token_store.hdf5 \
    --out_dir     synthetic_tokens_ratio \
    --strategy    ratio \
    --syn_ratio   0.5 \
    --temperature 0.3 \
    --device      auto

In [ ]:
# OPTION B: Full upsample (balances all classes to 4,168 each)
# Generates ~14,198 synthetic windows
# Time: ~20 minutes on MPS

!python generate_synthetic.py \
    --checkpoint  car_imu_checkpoints/best_model.pt \
    --token_store results_week1/token_store.hdf5 \
    --out_dir     synthetic_tokens \
    --strategy    upsample \
    --temperature 0.5 \
    --device      auto

---
## Step 4 — Evaluate: Real vs Augmented HAR

Runs LOSO (Leave-One-Subject-Out) cross-validation comparing:
- **[A]** Real-only tokens (64-d mean features)
- **[B]** Real + synthetic tokens

Baseline to beat: **Week 1 real-only F1 = 0.689**

In [ ]:
# Evaluate with ratio-strategy synthetic tokens
!python evaluate_augmented.py \
    --real_tokens results_week1/token_store.hdf5 \
    --syn_tokens  synthetic_tokens_ratio/synthetic_tokens.hdf5 \
    --out_dir     results_week2_ratio

In [ ]:
# Evaluate with full upsample synthetic tokens
!python evaluate_augmented.py \
    --real_tokens results_week1/token_store.hdf5 \
    --syn_tokens  synthetic_tokens/synthetic_tokens.hdf5 \
    --out_dir     results_week2

---
## Step 5 — Ablation: Compare Different Temperatures

If augmentation hurts, try lower temperature (less noise = more conservative synthetic data).

In [ ]:
import subprocess, json

temperatures = [0.1, 0.2, 0.3, 0.5]
results = {}

for temp in temperatures:
    print(f"\n{'='*50}")
    print(f"Temperature = {temp}")
    print('='*50)

    out_dir_syn  = f"synthetic_tokens_t{temp}"
    out_dir_eval = f"results_week2_t{temp}"

    # Generate
    !python generate_synthetic.py \
        --checkpoint  car_imu_checkpoints/best_model.pt \
        --token_store results_week1/token_store.hdf5 \
        --out_dir     {out_dir_syn} \
        --strategy    ratio \
        --syn_ratio   0.5 \
        --temperature {temp} \
        --device      auto

    # Evaluate
    !python evaluate_augmented.py \
        --real_tokens results_week1/token_store.hdf5 \
        --syn_tokens  {out_dir_syn}/synthetic_tokens.hdf5 \
        --out_dir     {out_dir_eval}

print("\nAblation complete!")

---
## Step 6 — View Saved Results

In [ ]:
import numpy as np
from car_imu_decoder import ACTIVITY_NAMES

res = np.load('results_week2/augmented_results.npy', allow_pickle=True).item()

print("=" * 55)
print("Week 2 Final Results")
print("=" * 55)

r_mlp = np.array(res['macro_real_mlp']).mean()
a_mlp = np.array(res['macro_aug_mlp']).mean()
delta  = a_mlp - r_mlp

print(f"\nReal only   MLP Macro-F1: {r_mlp:.3f}")
print(f"Augmented   MLP Macro-F1: {a_mlp:.3f}")
print(f"Delta:                   {delta:+.3f}")

print("\nPer-class F1 (MLP):")
print(f"  {'Class':<14} {'Real':>8} {'Aug':>8} {'Δ':>8}")
print("  " + "-" * 42)
for cls_id in range(6):
    r = np.mean(res['per_class_real'][cls_id]) if res['per_class_real'][cls_id] else float('nan')
    a = np.mean(res['per_class_aug'][cls_id])  if res['per_class_aug'][cls_id]  else float('nan')
    d = a - r
    print(f"  {ACTIVITY_NAMES[cls_id]:<14} {r:>8.3f} {a:>8.3f} {d:>+8.3f}")

---
## 📝 Week 2 Notes

### What worked
- CAR-IMU trained to Val MSE = **0.0157** (↓ from 1.0 at random init)
- Synthetic tokens had cosine similarity **> 0.85** with real tokens for all classes
- Standing class improved: **+0.014 F1** (rarest class with 471 windows)

### What didn't work (yet)
- Full upsample (14K synthetic) hurt overall F1 by **-0.024** — too many fake samples
- Sitting, Downstairs F1 dropped despite receiving synthetic data

### Next steps (Week 3)
1. Try `--strategy ratio --syn_ratio 0.5 --temperature 0.3`
2. Train model for 100 epochs with lower LR
3. Try KNN retrieval-based filtering: only keep synthetic samples close to real centroids
4. Write the ablation study section for the report

### Key numbers to report
| | F1 |
|---|---|
| Week 1 baseline (1028-d real) | 0.689 |
| Real only (64-d) | 0.661 |
| Real + CAR-IMU upsample | 0.637 |
| Real + CAR-IMU ratio 0.5 | TBD |